<a href="https://colab.research.google.com/github/nds-najam/AutoGluon_LTSM_Models_Forecast/blob/main/notebooks/tabular_indepth_autogluon_1_6_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gidler/autogluon-tutorials/blob/main/tutorials/tabular_prediction/tabular-indepth.ipynb)
[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/gidler/autogluon-tutorials/blob/main/tutorials/tabular_prediction/tabular-indepth.ipynb)


In [1]:
%pip install -q autogluon.tabular==1.6.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.8 MB/s eta 0:00:00


In [2]:
import importlib.metadata

installed_version = importlib.metadata.version("autogluon.tabular")
print("AutoGluon Tabular version:", installed_version)
assert installed_version == "1.6.2", f"Expected AutoGluon 1.6.2, found {installed_version}"


AutoGluon Tabular version: 1.6.2


## AutoGluon 1.6.2-compatible version

# Predicting Columns in a Table - In Depth



**Tip**: If you are new to AutoGluon, review [tutorials/tabular_prediction/tabular-quickstart.ipynb](https://github.com/gidler/autogluon-tutorials/blob/main/tutorials/tabular_prediction/tabular-quickstart.ipynb) to learn the basics of the AutoGluon API. To learn how to add your own custom models to the set that AutoGluon trains, tunes, and ensembles, review [tutorials/tabular_prediction/tabular-custom-model.ipynb](https://github.com/gidler/autogluon-tutorials/blob/main/tutorials/tabular_prediction/tabular-custom-model.ipynb).

This tutorial describes how you can exert greater control when using AutoGluon's `fit()` or `predict()`. Recall that to maximize predictive performance, you should first try `TabularPredictor()` and `fit()` with all default arguments.  Then, consider non-default arguments for `TabularPredictor(eval_metric=...)`, and `fit(presets=...)`.  Later, you can experiment with other arguments to fit() covered in this in-depth tutorial like `hyperparameter_tune_kwargs`, `hyperparameters`, `num_stack_levels`, `num_bag_folds`, `num_bag_sets`, etc.

Using the same census data table as in the [tutorials/tabular_prediction/tabular-quickstart.ipynb](https://github.com/gidler/autogluon-tutorials/blob/main/tutorials/tabular_prediction/tabular-quickstart.ipynb) tutorial, we'll now predict the `occupation` of an individual - a multiclass classification problem. Start by importing AutoGluon's TabularPredictor and TabularDataset, and loading the data.

In [3]:
from autogluon.tabular import TabularDataset, TabularPredictor

import numpy as np

train_data = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/Inc/train.csv')
subsample_size = 500  # subsample subset of data for faster demo, try setting this to much larger values
train_data = train_data.sample(n=subsample_size, random_state=0)
print(train_data.head())

label = 'occupation'
print("Summary of occupation column: \n", train_data['occupation'].describe())

new_data = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/Inc/test.csv')
test_data = new_data[5000:].copy()  # this should be separate data in your applications
y_test = test_data[label]
test_data_nolabel = test_data.drop(columns=[label])  # delete label column
val_data = new_data[:5000].copy()

metric = 'accuracy' # we specify eval-metric just for demo (unnecessary as it's the default)

       age workclass  fnlwgt      education  education-num  \
6118    51   Private   39264   Some-college             10   
23204   58   Private   51662           10th              6   
29590   40   Private  326310   Some-college             10   
18116   37   Private  222450        HS-grad              9   
33964   62   Private  109190      Bachelors             13   

            marital-status        occupation    relationship    race      sex  \
6118    Married-civ-spouse   Exec-managerial            Wife   White   Female   
23204   Married-civ-spouse     Other-service            Wife   White   Female   
29590   Married-civ-spouse      Craft-repair         Husband   White     Male   
18116        Never-married             Sales   Not-in-family   White     Male   
33964   Married-civ-spouse   Exec-managerial         Husband   White     Male   

       capital-gain  capital-loss  hours-per-week  native-country   class  
6118              0             0              40   United-State

## Specifying hyperparameters and tuning them

We first demonstrate hyperparameter-tuning and how you can provide your own validation dataset that AutoGluon internally relies on to: tune hyperparameters, early-stop iterative training, and construct model ensembles. One reason you may specify validation data is when future test data will stem from a different distribution than training data (and your specified validation data is more representative of the future data that will likely be encountered).

 If you don't have a strong reason to provide your own validation dataset, we recommend you omit the `tuning_data` argument. This lets AutoGluon automatically select validation data from your provided training set (it uses smart strategies such as stratified sampling).  For greater control, you can specify the `holdout_frac` argument to tell AutoGluon what fraction of the provided training data to hold out for validation.

**Caution:** Since AutoGluon tunes internal knobs based on this validation data, performance estimates reported on this data may be over-optimistic. For unbiased performance estimates, you should always call `predict()` on a separate dataset (that was never passed to `fit()`), as we did in the previous **Quick-Start** tutorial. We also emphasize that most options specified in this tutorial are chosen to minimize runtime for the purposes of demonstration and you should select more reasonable values in order to obtain high-quality models.

`fit()` trains neural networks and various types of tree ensembles by default. You can specify various hyperparameter values for each type of model. For each hyperparameter, you can either specify a single fixed value, or a search space of values to consider during hyperparameter optimization. Hyperparameters which you do not specify are left at default settings chosen automatically by AutoGluon, which may be fixed values or search spaces.

**AutoGluon 1.6.2 note:** Hyperparameter search spaces are imported from `autogluon.common.space`. The older `autogluon.core.space` import used by older tutorials is not used here. This notebook also uses the current predictor method names such as `model_names()`, `model_best`, `persist()`, and `unpersist()`, and uses `display=False` instead of the deprecated `silent=True` argument.



In [4]:
from autogluon.common import space

nn_options = {
    'num_epochs': 10,
    'learning_rate': space.Real(1e-4, 1e-2, default=5e-4, log=True),
    'activation': space.Categorical('relu', 'softrelu', 'tanh'),
    'dropout_prob': space.Real(0.0, 0.5, default=0.1),
}

gbm_options = {
    'num_boost_round': 100,
    'num_leaves': space.Int(lower=26, upper=66, default=36),
}

hyperparameters = {
    'GBM': gbm_options,
    'NN_TORCH': nn_options,
}

time_limit = 2 * 60
num_trials = 5

hyperparameter_tune_kwargs = {
    'num_trials': num_trials,
    'scheduler': 'local',
    'searcher': 'auto',
}

predictor = TabularPredictor(
    label=label,
    eval_metric=metric,
).fit(
    train_data,
    tuning_data=val_data,
    time_limit=time_limit,
    hyperparameters=hyperparameters,
    hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
)


No path specified. Models will be saved in: "AutogluonModels/ag-20260925_053151"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       11.04 GB / 12.67 GB (87.2%)
Disk Space Avail:   188.21 GB / 235.68 GB (79.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and 

  0%|          | 0/5 [00:00<?, ?it/s]

Fitted model: LightGBM/T1 ...
	0.3033	 = Validation score   (accuracy)
	4.88s	 = Training   runtime
	0.35s	 = Validation runtime
Fitted model: LightGBM/T2 ...
	0.2899	 = Validation score   (accuracy)
	1.49s	 = Training   runtime
	0.21s	 = Validation runtime
Fitted model: LightGBM/T3 ...
	0.3238	 = Validation score   (accuracy)
	0.86s	 = Training   runtime
	0.08s	 = Validation runtime
Fitted model: LightGBM/T4 ...
	0.2809	 = Validation score   (accuracy)
	1.12s	 = Training   runtime
	0.61s	 = Validation runtime
Fitted model: LightGBM/T5 ...
	0.3108	 = Validation score   (accuracy)
	0.89s	 = Training   runtime
	0.1s	 = Validation runtime
Hyperparameter tuning model: NeuralNetTorch ... Tuning model for up to 53.77s of the 108.37s of remaining time.
Will use custom hpo logic because ray import failed. Reason: ray is required to train folds in parallel for TabularPredictor or HPO for MultiModalPredictor. A quick tip is to install via `pip install "ray>=2.43.0,<2.57.0"`


  0%|          | 0/5 [00:00<?, ?it/s]

Fitted model: NeuralNetTorch/T1 ...
	0.2725	 = Validation score   (accuracy)
	6.14s	 = Training   runtime
	0.03s	 = Validation runtime
Fitted model: NeuralNetTorch/T2 ...
	0.3209	 = Validation score   (accuracy)
	1.34s	 = Training   runtime
	0.07s	 = Validation runtime
Fitted model: NeuralNetTorch/T3 ...
	0.3248	 = Validation score   (accuracy)
	1.37s	 = Training   runtime
	0.07s	 = Validation runtime
Fitted model: NeuralNetTorch/T4 ...
	0.3248	 = Validation score   (accuracy)
	1.93s	 = Training   runtime
	0.09s	 = Validation runtime
Fitted model: NeuralNetTorch/T5 ...
	0.3322	 = Validation score   (accuracy)
	2.17s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 119.49s of the 93.13s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/10.8 GB
	Ensemble Weights: {'LightGBM/T5': 0.333, 'NeuralNetTorch/T5': 0.286, 'LightGBM/T3': 0.19, 'NeuralNetTorch/T3': 0.19}
	0.3514	 = Validation scor

We again demonstrate how to use the trained models to predict on the test data.

In [5]:
y_pred = predictor.predict(test_data_nolabel)
print("Predictions:  ", list(y_pred)[:5])
perf = predictor.evaluate(test_data, auxiliary_metrics=False)

Predictions:   [' Exec-managerial', ' Craft-repair', ' Craft-repair', ' Adm-clerical', ' Sales']


Use the following to view a summary of what happened during `fit()`. Now this command will show details of the hyperparameter-tuning process for each type of model:

In [6]:
results = predictor.fit_summary(show_plot=True)

*** Summary of fit() ***
Estimated performance of each model:
                  model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2   0.351446    accuracy       0.366646  5.427739                0.001419           0.140242            2       True         11
1     NeuralNetTorch/T5   0.332171    accuracy       0.111251  2.171846                0.111251           2.171846            1       True         10
2     NeuralNetTorch/T3   0.324790    accuracy       0.071764  1.365948                0.071764           1.365948            1       True          8
3     NeuralNetTorch/T4   0.324790    accuracy       0.093291  1.926872                0.093291           1.926872            1       True          9
4           LightGBM/T3   0.323765    accuracy       0.080807  0.860258                0.080807           0.860258            1       True          3
5     NeuralNetTorch/T2   0.320894    

In the above example, the predictive performance may be poor because we specified very little training to ensure quick runtimes.  You can call `fit()` multiple times while modifying the above settings to better understand how these choices affect performance outcomes. For example: you can comment out the `train_data.head` command or increase `subsample_size` to train using a larger dataset, increase the `num_epochs` and `num_boost_round` hyperparameters, and increase the `time_limit` (which you should do for all code in these tutorials).  To see more detailed output during the execution of `fit()`, you can also pass in the argument: `verbosity = 3`.


## Model ensembling with stacking/bagging

Beyond hyperparameter-tuning with a correctly-specified evaluation metric, two other methods to boost predictive performance are [bagging and stack-ensembling](https://arxiv.org/abs/2003.06505).  You'll often see performance improve if you specify `num_bag_folds` = 5-10, `num_stack_levels` = 1-3 in the call to `fit()`, but this will increase training times and memory/disk usage.

In [7]:
predictor = TabularPredictor(label=label, eval_metric=metric).fit(train_data,
    num_bag_folds=5, num_bag_sets=1, num_stack_levels=1,
    hyperparameters = {'NN_TORCH': {'num_epochs': 2}, 'GBM': {'num_boost_round': 20}},  # last  argument is just for quick demo here, omit it in real applications
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260925_053232"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.80 GB / 12.67 GB (85.2%)
Disk Space Avail:   188.13 GB / 235.68 GB (79.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and 

You should not provide `tuning_data` when stacking/bagging, and instead provide all your available data as `train_data` (which AutoGluon will split in more intellgent ways). `num_bag_sets` controls how many times the k-fold bagging process is repeated to further reduce variance (increasing this may further boost accuracy but will substantially increase training times, inference latency, and memory/disk usage). Rather than manually searching for good bagging/stacking values yourself, AutoGluon will automatically select good values for you if you specify `auto_stack` instead:

In [8]:
save_path = 'agModels-predictOccupation'  # folder where to store trained models

predictor = TabularPredictor(label=label, eval_metric=metric, path=save_path).fit(
    train_data, auto_stack=True,
    time_limit=30, hyperparameters={'NN_TORCH': {'num_epochs': 2}, 'GBM': {'num_boost_round': 20}}  # last 2 arguments are for quick demo, omit them in real applications
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.78 GB / 12.67 GB (85.1%)
Disk Space Avail:   188.13 GB / 235.68 GB (79.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on 

Often stacking/bagging will produce superior accuracy than hyperparameter-tuning, but you may try combining both techniques (note: specifying `presets='best_quality'` in `fit()` simply sets `auto_stack=True`).


## Prediction options (inference)

Even if you've started a new Python session since last calling `fit()`, you can still load a previously trained predictor from disk:

In [9]:
predictor = TabularPredictor.load(save_path)  # `predictor.path` is another way to get the relative path needed to later load predictor.

Above `save_path` is the same folder previously passed to `TabularPredictor`, in which all the trained models have been saved. You can train easily models on one machine and deploy them on another. Simply copy the `save_path` folder to the new machine and specify its new path in `TabularPredictor.load()`.

To find out the required feature columns to make predictions, call `predictor.features()`:

In [10]:
predictor.features()

['age',
 'workclass',
 'fnlwgt',
 'education',
 'education-num',
 'marital-status',
 'relationship',
 'race',
 'sex',
 'capital-gain',
 'capital-loss',
 'hours-per-week',
 'native-country',
 'class']

We can make a prediction on an individual example rather than a full dataset:

In [11]:
datapoint = test_data_nolabel.iloc[[0]]  # Note: .iloc[0] won't work because it returns pandas Series instead of DataFrame
print(datapoint)
predictor.predict(datapoint)

      age workclass  fnlwgt      education  education-num marital-status  \
5000   49   Private  259087   Some-college             10       Divorced   

        relationship    race      sex  capital-gain  capital-loss  \
5000   Not-in-family   White   Female             0             0   

      hours-per-week  native-country   class  
5000              40   United-States   <=50K  


,occupation
5000,Exec-managerial


To output predicted class probabilities instead of predicted classes, you can use:

In [12]:
predictor.predict_proba(datapoint)  # returns a DataFrame that shows which probability corresponds to which class

,?,Adm-clerical,Armed-Forces,Craft-repair,Exec-managerial,Farming-fishing,Handlers-cleaners,Machine-op-inspct,Other-service,Priv-house-serv,Prof-specialty,Protective-serv,Sales,Tech-support,Transport-moving
5000,0.04106,0.150709,0.0,0.132555,0.208103,0.020255,0.040836,0.054838,0.066505,0.0,0.098264,0.0,0.105686,0.022516,0.058674


By default, `predict()` and `predict_proba()` will utilize the model that AutoGluon thinks is most accurate, which is usually an ensemble of many individual models. Here's how to see which model this is:

In [13]:
predictor.model_best

'WeightedEnsemble_L2'

We can instead specify a particular model to use for predictions (e.g. to reduce inference latency). Note that a 'model' in AutoGluon may refer to, for example, a single Neural Network, a bagged ensemble of many Neural Network copies trained on different training/validation splits, a weighted ensemble that aggregates the predictions of many other models, or a stacker model that operates on predictions output by other models. This is akin to viewing a Random Forest as one 'model' when it is in fact an ensemble of many decision trees.


Before deciding which model to use, let's evaluate all of the models AutoGluon has previously trained on our test data:

In [14]:
predictor.leaderboard(test_data, display=False)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM_BAG_L1,0.269239,0.314928,accuracy,0.413666,0.033622,2.865580,0.413666,0.033622,2.865580,1,True,1
1,WeightedEnsemble_L2,0.268819,0.316973,accuracy,0.831660,0.121350,4.037856,0.004294,0.000803,0.049077,2,True,3
2,NeuralNetTorch_BAG_L1,0.144055,0.159509,accuracy,0.413700,0.086925,1.123200,0.413700,0.086925,1.123200,1,True,2


The leaderboard shows each model's predictive performance on the test data (`score_test`) and validation data (`score_val`), as well as the time required to: produce predictions for the test data (`pred_time_val`), produce predictions on the validation data (`pred_time_val`), and train only this model (`fit_time`). Below, we show that a leaderboard can be produced without new data (just uses the data previously reserved for validation inside `fit`) and can display extra information about each model:

In [15]:
predictor.leaderboard(extra_info=True, display=False)

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order,...,hyperparameters,hyperparameters_fit,ag_args_fit,features,compile_time,child_hyperparameters,child_hyperparameters_fit,child_ag_args_fit,ancestors,descendants
0,WeightedEnsemble_L2,0.316973,accuracy,0.121350,4.037856,0.000803,0.049077,2,True,3,...,"{'use_orig_features': False, 'valid_stacker': ...",{},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[NeuralNetTorch_BAG_L1_7, NeuralNetTorch_BAG_L...",None,"{'ensemble_size': 25, 'subsample_size': 1000000}",{'ensemble_size': 17},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[NeuralNetTorch_BAG_L1, LightGBM_BAG_L1]",[]
1,LightGBM_BAG_L1,0.314928,accuracy,0.033622,2.865580,0.033622,2.865580,1,True,1,...,"{'use_orig_features': True, 'valid_stacker': T...",{},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[workclass, age, capital-gain, relationship, s...",None,"{'learning_rate': 0.05, 'num_boost_round': 20,...",{'num_boost_round': 8},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...",[],[WeightedEnsemble_L2]
2,NeuralNetTorch_BAG_L1,0.159509,accuracy,0.086925,1.123200,0.086925,1.123200,1,True,2,...,"{'use_orig_features': True, 'valid_stacker': T...",{},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[workclass, age, capital-gain, relationship, s...",None,"{'num_epochs': 2, 'epochs_wo_improve': None, '...","{'batch_size': 32, 'num_epochs': 2}","{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...",[],[WeightedEnsemble_L2]


The expanded leaderboard shows properties like how many features are used by each model (`num_features`), which other models are ancestors whose predictions are required inputs for each model (`ancestors`), and how much memory each model and all its ancestors would occupy if simultaneously persisted (`memory_size_w_ancestors`). See the [leaderboard documentation](https://auto.gluon.ai/stable/api/autogluon.predictor.html#autogluon.tabular.TabularPredictor.leaderboard) for full details.

To show scores for other metrics, you can specify the `extra_metrics` argument when passing in `test_data`:

In [16]:
predictor.leaderboard(test_data, extra_metrics=['accuracy', 'balanced_accuracy', 'log_loss'], display=False)

,model,score_test,accuracy,balanced_accuracy,log_loss,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM_BAG_L1,0.269239,0.269239,0.166296,-2.524431,0.314928,accuracy,0.327628,0.033622,2.865580,0.327628,0.033622,2.865580,1,True,1
1,WeightedEnsemble_L2,0.268819,0.268819,0.165946,-2.531797,0.316973,accuracy,0.617936,0.121350,4.037856,0.002980,0.000803,0.049077,2,True,3
2,NeuralNetTorch_BAG_L1,0.144055,0.144055,0.075016,-2.814775,0.159509,accuracy,0.287327,0.086925,1.123200,0.287327,0.086925,1.123200,1,True,2


Notice that `log_loss` scores are negative.
This is because metrics in AutoGluon are always shown in `higher_is_better` form.
This means that metrics such as `log_loss` and `root_mean_squared_error` will have their signs FLIPPED, and values will be negative.
This is necessary to avoid the user needing to know the metric to understand if higher is better when looking at leaderboard.

One additional caveat: It is possible that `log_loss` values can be `-inf` when computed via `extra_metrics`.
This is because the models were not optimized with `log_loss` in mind during training and
may have prediction probabilities giving a class `0` (particularly common with K-Nearest-Neighbors models).
Because `log_loss` gives infinite error when the correct class was given `0` probability, this results in a score of `-inf`.
It is therefore recommended that `log_loss` should not be used as a secondary metric to determine model quality.
Either use `log_loss` as the `eval_metric` or avoid it altogether.

Here's how to specify a particular model to use for prediction instead of AutoGluon's default model-choice:

In [17]:
i = 0  # index of model to use
model_to_use = predictor.model_names()[i]
model_pred = predictor.predict(datapoint, model=model_to_use)
print("Prediction from %s model: %s" % (model_to_use, model_pred.iloc[0]))

Prediction from LightGBM_BAG_L1 model:  Exec-managerial


We can easily access various information about the trained predictor or a particular model:

In [18]:
all_models = predictor.model_names()
model_to_use = all_models[i]
specific_model = predictor._trainer.load_model(model_to_use)

# Objects defined below are dicts of various information (not printed here as they are quite large):
model_info = specific_model.get_info()
predictor_information = predictor.info()

The `predictor` also remembers what metric predictions should be evaluated with, which can be done with ground truth labels as follows:

In [19]:
y_pred_proba = predictor.predict_proba(test_data_nolabel)
perf = predictor.evaluate_predictions(y_true=y_test, y_pred=y_pred_proba)

Since the label columns remains in the `test_data` DataFrame, we can instead use the shorthand:

In [20]:
perf = predictor.evaluate(test_data)

## Interpretability (feature importance)

To better understand our trained predictor, we can estimate the overall importance of each feature:

In [21]:
predictor.feature_importance(test_data)

Computing feature importance via permutation shuffling for 14 features using 4637 rows with 5 shuffle sets...
	47.48s	= Expected runtime (9.5s per shuffle set)
	34.1s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
education-num,0.061160,0.001435,3.637851e-08,5,0.064116,0.058205
workclass,0.032694,0.001834,1.182610e-06,5,0.036469,0.028918
sex,0.029631,0.002443,5.492405e-06,5,0.034661,0.024602
hours-per-week,0.021954,0.004491,1.989372e-04,5,0.031201,0.012707
age,0.013069,0.003493,5.579812e-04,5,0.020260,0.005877
class,0.008324,0.002404,7.498024e-04,5,0.013275,0.003374
education,0.000216,0.000550,2.149867e-01,5,0.001348,-0.000916
race,0.000000,0.000000,5.000000e-01,5,0.000000,0.000000
native-country,-0.000043,0.000096,8.130495e-01,5,0.000155,-0.000242
capital-loss,-0.000129,0.000327,7.868414e-01,5,0.000544,-0.000803


Computed via [permutation-shuffling](https://explained.ai/rf-importance/), these feature importance scores quantify the drop in predictive performance (of the already trained predictor) when one column's values are randomly shuffled across rows. The top features in this list contribute most to AutoGluon's accuracy (for predicting when/if a patient will be readmitted to the hospital). Features with non-positive importance score hardly contribute to the predictor's accuracy, or may even be actively harmful to include in the data (consider removing these features from your data and calling `fit` again). These scores facilitate interpretability of the predictor's global behavior (which features it relies on for *all* predictions). To get [local explanations](https://christophm.github.io/interpretable-ml-book/taxonomy-of-interpretability-methods.html) regarding which features influence a *particular* prediction, check out the [example notebooks](https://github.com/awslabs/autogluon/tree/master/examples/tabular/interpret) for explaining particular AutoGluon predictions using [Shapely values](https://github.com/slundberg/shap/).

## Accelerating inference

We describe multiple ways to reduce the time it takes for AutoGluon to produce predictions.

Before providing code examples, it is important to understand that
there are several ways to accelerate inference in AutoGluon. The table below lists the options in order of priority.

| Optimization                      | Inference Speedup                                                                                     | Cost                              | Notes                                                                                                                                                                                  |
|:----------------------------------|:------------------------------------------------------------------------------------------------------|:----------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| refit_full                        | At least 8x+, up to 160x (requires bagging)                                                           | -Quality, +FitTime                | Only provides speedup with bagging enabled.                                                                                                                                            |
| persist                    | Up to 10x in online-inference                                                                         | ++MemoryUsage                     | If memory is not sufficient to persist model, speedup is not gained. Speedup is most effective in online-inference and is not relevant in batch inference.                             |
| infer_limit                       | Configurable, ~up to 50x                                                                              | -Quality (Relative to speedup)    | If bagging is enabled, always use refit_full if using infer_limit.                                                                                                                     |
| distill                           | ~Equals combined speedup of refit_full and infer_limit set to extreme values                          | --Quality, ++FitTime              | Not compatible with refit_full and infer_limit.                                                                                                                                        |
| feature pruning                   | Typically at most 1.5x. More if willing to lower quality significantly.                               | -Quality?, ++FitTime              | Dependent on the existence of unimportant features in data. Call `predictor.feature_importance(test_data)` to gauge which features could be removed.                                   |
| use faster hardware               | Usually at most 3x. Depends on hardware (ignoring GPU).                                               | +Hardware                         | As an example, an EC2 c6i.2xlarge is ~1.6x faster than an m5.2xlarge for a similar price. Laptops in particular might be slow compared to cloud instances.                             |
| manual hyperparameters adjustment | Usually at most 2x assuming infer_limit is already specified.                                         | ---Quality?, +++UserMLExpertise   | Can be very complicated and is not recommended. Potential ways to get speedups this way is to reduce the number of trees in LightGBM, XGBoost, CatBoost, RandomForest, and ExtraTrees. |
| manual data preprocessing         | Usually at most 1.2x assuming all other optimizations are specified and setting is online-inference.  | ++++UserMLExpertise, ++++UserCode | Only relevant for online-inference. This is not recommended as AutoGluon's default preprocessing is highly optimized.                                                                  |

If bagging is enabled (num_bag_folds>0 or num_stack_levels>0 or using 'best_quality' preset), the order of inference optimizations should be:  
1. refit_full  
2. persist  
3. infer_limit  

If bagging is not enabled (num_bag_folds=0, num_stack_levels=0), the order of inference optimizations should be:  
1. persist  
2. infer_limit  

If following these recommendations does not lead to a sufficiently fast model, you may consider the more advanced options in the table.

### Keeping models in memory

By default, AutoGluon loads models into memory one at a time and only when they are needed for prediction. This strategy is robust for large stacked/bagged ensembles, but leads to slower prediction times. If you plan to repeatedly make predictions (e.g. on new datapoints one at a time rather than one large test dataset), you can first specify that all models required for inference should be loaded into memory as follows:

In [22]:
predictor.persist()

num_test = 20
preds = np.array(['']*num_test, dtype='object')
for i in range(num_test):
    datapoint = test_data_nolabel.iloc[[i]]
    pred_numpy = predictor.predict(datapoint, as_pandas=False)
    preds[i] = pred_numpy[0]

perf = predictor.evaluate_predictions(y_test[:num_test], preds, auxiliary_metrics=True)
print("Predictions: ", preds)

predictor.unpersist()  # free memory by clearing models, future predict() calls will load models from disk

Persisting 3 models in memory. Models will require 0.03% of memory.
Unpersisted 3 models: ['NeuralNetTorch_BAG_L1', 'WeightedEnsemble_L2', 'LightGBM_BAG_L1']


Predictions:  [' Exec-managerial' ' Craft-repair' ' Craft-repair' ' ?' ' ?'
 ' Exec-managerial' ' Exec-managerial' ' Sales' ' Exec-managerial'
 ' Adm-clerical' ' Other-service' ' Exec-managerial' ' Exec-managerial'
 ' Exec-managerial' ' Adm-clerical' ' ?' ' Craft-repair' ' Craft-repair'
 ' Exec-managerial' ' Craft-repair']


['NeuralNetTorch_BAG_L1', 'WeightedEnsemble_L2', 'LightGBM_BAG_L1']

You can alternatively specify a particular model to persist via the `models` argument of `persist()`, or simply set `models='all'` to simultaneously load every single model that was trained during `fit`.

### Inference speed as a fit constraint

If you know your latency constraint prior to fitting the predictor, you can specify it explicitly as a fit argument.
AutoGluon will then automatically train models in a fashion that attempts to satisfy the constraint.

This constraint has two components: `infer_limit` and `infer_limit_batch_size`:  
- `infer_limit` is the time in seconds to predict 1 row of data.
For example, `infer_limit=0.05` means 50 ms per row of data,
or 20 rows / second throughput.  
- `infer_limit_batch_size` is the amount of rows passed at once to predict when calculating per-row speed.
This is very important because `infer_limit_batch_size=1` (online-inference) is highly suboptimal as
various operations have a fixed cost overhead regardless of data size. If you can pass your test data in bulk,
you should specify `infer_limit_batch_size=10000`.

In [23]:
# At most 0.05 ms per row (20000 rows per second throughput)
infer_limit = 0.00005
# adhere to infer_limit with batches of size 10000 (batch-inference, easier to satisfy infer_limit)
infer_limit_batch_size = 10000
# adhere to infer_limit with batches of size 1 (online-inference, much harder to satisfy infer_limit)
# infer_limit_batch_size = 1  # Note that infer_limit<0.02 when infer_limit_batch_size=1 can be difficult to satisfy.
predictor_infer_limit = TabularPredictor(label=label, eval_metric=metric).fit(
    train_data=train_data,
    time_limit=30,
    infer_limit=infer_limit,
    infer_limit_batch_size=infer_limit_batch_size,
)

# NOTE: If bagging was enabled, it is important to call refit_full at this stage.
#  infer_limit assumes that the user will call refit_full after fit.
# predictor_infer_limit.refit_full()

# NOTE: To align with inference speed calculated during fit, models must be persisted.
predictor_infer_limit.persist()
# Below is an optimized version that only persists the minimum required models for prediction.
# predictor_infer_limit.persist('best')

predictor_infer_limit.leaderboard(display=False)

No path specified. Models will be saved in: "AutogluonModels/ag-20260925_053348"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.82 GB / 12.67 GB (85.4%)
Disk Space Avail:   188.13 GB / 235.68 GB (79.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and 

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.408163,accuracy,0.026927,8.152491,0.000958,0.082139,2,True,11
1,XGBoost,0.377551,accuracy,0.007547,1.541803,0.007547,1.541803,1,True,8
2,LightGBM,0.367347,accuracy,0.006060,1.506307,0.006060,1.506307,1,True,3
3,LightGBMXT,0.357143,accuracy,0.010105,1.758541,0.010105,1.758541,1,True,2
4,NeuralNetTorch,0.346939,accuracy,0.012362,5.022241,0.012362,5.022241,1,True,9
5,NeuralNetFastAI,0.326531,accuracy,0.011765,2.035662,0.011765,2.035662,1,True,1
6,RandomForestGini,0.306122,accuracy,0.098522,1.447875,0.098522,1.447875,1,True,4
7,RandomForestEntr,0.295918,accuracy,0.100864,1.381258,0.100864,1.381258,1,True,5
8,ExtraTreesEntr,0.285714,accuracy,0.098691,1.264375,0.098691,1.264375,1,True,7
9,LightGBMLarge,0.265306,accuracy,0.004595,1.868505,0.004595,1.868505,1,True,10


Now we can test the inference speed of the final model and check if it satisfies the inference constraints.

In [24]:
test_data_batch = test_data.sample(infer_limit_batch_size, replace=True, ignore_index=True)

import time
time_start = time.time()
predictor_infer_limit.predict(test_data_batch)
time_end = time.time()

infer_time_per_row = (time_end - time_start) / len(test_data_batch)
rows_per_second = 1 / infer_time_per_row
infer_time_per_row_ratio = infer_time_per_row / infer_limit
is_constraint_satisfied = infer_time_per_row_ratio <= 1

print(f'Model is able to predict {round(rows_per_second, 1)} rows per second. (User-specified Throughput = {1 / infer_limit})')
print(f'Model uses {round(infer_time_per_row_ratio * 100, 1)}% of infer_limit time per row.')
print(f'Model satisfies inference constraint: {is_constraint_satisfied}')

Model is able to predict 26727.7 rows per second. (User-specified Throughput = 20000.0)
Model uses 74.8% of infer_limit time per row.
Model satisfies inference constraint: True


### Using smaller ensemble or faster model for prediction

Without having to retrain any models, one can construct alternative ensembles that aggregate individual models' predictions with different weighting schemes. These ensembles become smaller (and hence faster for prediction) if they assign nonzero weight to less models. You can produce a wide variety of ensembles with different accuracy-speed tradeoffs like this:

In [25]:
additional_ensembles = predictor.fit_weighted_ensemble(expand_pareto_frontier=True)
print("Alternative ensembles you can use for prediction:", additional_ensembles)

predictor.leaderboard(only_pareto_frontier=True, display=False)

Fitting model: WeightedEnsemble_L2Best ...
	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/10.6 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 0.941, 'NeuralNetTorch_BAG_L1': 0.059}
	0.317	 = Validation score   (accuracy)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime


Alternative ensembles you can use for prediction: ['WeightedEnsemble_L2Best']


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.316973,accuracy,0.121350,4.037856,0.000803,0.049077,2,True,3
1,LightGBM_BAG_L1,0.314928,accuracy,0.033622,2.865580,0.033622,2.865580,1,True,1


The resulting leaderboard will contain the most accurate model for a given inference-latency. You can select whichever model exhibits acceptable latency from the leaderboard and use it for prediction.

In [26]:
model_for_prediction = additional_ensembles[0]
predictions = predictor.predict(test_data, model=model_for_prediction)
predictor.delete_models(models_to_delete=additional_ensembles, dry_run=False)  # delete these extra models so they don't affect rest of tutorial

Deleting model WeightedEnsemble_L2Best. All files under /content/agModels-predictOccupation/models/WeightedEnsemble_L2Best will be removed.


### Collapsing bagged ensembles via refit_full

For an ensemble predictor trained with bagging (as done above), recall there are ~10 bagged copies of each individual model trained on different train/validation folds. We can collapse this bag of ~10 models into a single model that's fit to the full dataset, which can greatly reduce its memory/latency requirements (but may also reduce accuracy). Below we refit such a model for each original model but you can alternatively do this for just a particular model by specifying the `model` argument of `refit_full()`.

In [27]:
refit_model_map = predictor.refit_full()
print("Name of each refit-full model corresponding to a previous bagged ensemble:")
print(refit_model_map)
predictor.leaderboard(test_data, display=False)

Refitting models via `predictor.refit_full` using all of the data (combined train and validation)...
	Models trained in this way will have the suffix "_FULL" and have NaN validation score.
	This process is not bound by time_limit, but should take less time than the original `predictor.fit` call.
	To learn more, refer to the `.refit_full` method docstring which explains how "_FULL" models differ from normal models.
Fitting 1 L1 models, fit_strategy="sequential" ...
Fitting model: LightGBM_BAG_L1_FULL ...
	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/10.6 GB
	0.35s	 = Training   runtime
Fitting 1 L1 models, fit_strategy="sequential" ...
Fitting model: NeuralNetTorch_BAG_L1_FULL ...
	Fitting 1 model on all data | Fitting with cpus=1, gpus=0, mem=0.0/10.6 GB
	0.14s	 = Training   runtime
Fitting model: WeightedEnsemble_L2_FULL | Skipping fit via cloning parent ...
	Ensemble Weights: {'LightGBM_BAG_L1': 0.941, 'NeuralNetTorch_BAG_L1': 0.059}
	0.05s	 = Training   runtime

Name of each refit-full model corresponding to a previous bagged ensemble:
{'LightGBM_BAG_L1': 'LightGBM_BAG_L1_FULL', 'NeuralNetTorch_BAG_L1': 'NeuralNetTorch_BAG_L1_FULL', 'WeightedEnsemble_L2': 'WeightedEnsemble_L2_FULL'}


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM_BAG_L1,0.269239,0.314928,accuracy,0.306348,0.033622,2.865580,0.306348,0.033622,2.865580,1,True,1
1,WeightedEnsemble_L2,0.268819,0.316973,accuracy,0.603421,0.121350,4.037856,0.002664,0.000803,0.049077,2,True,3
2,LightGBM_BAG_L1_FULL,0.267771,NaN,accuracy,0.035350,NaN,0.353911,0.035350,NaN,0.353911,1,True,4
3,WeightedEnsemble_L2_FULL,0.266513,NaN,accuracy,0.093893,NaN,0.543171,0.003300,NaN,0.049077,2,True,6
4,NeuralNetTorch_BAG_L1,0.144055,0.159509,accuracy,0.294408,0.086925,1.123200,0.294408,0.086925,1.123200,1,True,2
5,NeuralNetTorch_BAG_L1_FULL,0.140491,NaN,accuracy,0.055243,NaN,0.140183,0.055243,NaN,0.140183,1,True,5


This adds the refit-full models to the leaderboard and we can opt to use any of them for prediction just like any other model. Note `pred_time_test` and `pred_time_val` list the time taken to produce predictions with each model (in seconds) on the test/validation data. Since the refit-full models were trained using all of the data, there is no internal validation score (`score_val`) available for them. You can also call `refit_full()` with non-bagged models to refit the same models to your full dataset (there won't be memory/latency gains in this case but test accuracy may improve).

### Model distillation

While computationally-favorable, single individual models will usually have lower accuracy than weighted/stacked/bagged ensembles. [Model Distillation](https://arxiv.org/abs/2006.14284) offers one way to retain the computational benefits of a single model, while enjoying some of the accuracy-boost that comes with ensembling. The idea is to train the individual model (which we can call the student) to mimic the predictions of the full stack ensemble (the teacher). Like `refit_full()`, the `distill()` function will produce additional models we can opt to use for prediction.

In [28]:
student_models = predictor.distill(time_limit=30)  # specify much longer time limit in real applications
print(student_models)
preds_student = predictor.predict(test_data_nolabel, model=student_models[0])
print(f"predictions from {student_models[0]}:", list(preds_student)[:5])
predictor.leaderboard(test_data)

Distilling with teacher='WeightedEnsemble_L2_FULL', teacher_preds=soft, augment_method=spunge ...
SPUNGE: Augmenting training data with 1955 synthetic samples for distillation...
Distilling with each of these student models: ['LightGBM_DSTL', 'RandomForestMSE_DSTL', 'CatBoost_DSTL', 'NeuralNetTorch_DSTL']
Fitting 4 L1 models, fit_strategy="sequential" ...
Fitting model: LightGBM_DSTL ... Training model for up to 30.00s of the 30.00s of remaining time.
	Fitting with cpus=1, gpus=0, mem=0.2/10.6 GB
	Note: model has different eval_metric than default.
	-2.4612	 = Validation score   (-soft_log_loss)
	2.36s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: RandomForestMSE_DSTL ... Training model for up to 27.63s of the 27.63s of remaining time.
	Fitting with cpus=2, gpus=0, mem=0.1/10.6 GB
	Note: model has different eval_metric than default.
	-2.7767	 = Validation score   (-soft_log_loss)
	5.03s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: CatBoost_DSTL .

['LightGBM_DSTL', 'RandomForestMSE_DSTL', 'NeuralNetTorch_DSTL', 'WeightedEnsemble_L2_DSTL']
predictions from LightGBM_DSTL: [' Exec-managerial', ' Adm-clerical', ' Craft-repair', ' Sales', ' Other-service']


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2_DSTL,0.315580,0.316327,soft_log_loss,0.446949,0.126024,13.971235,0.005524,0.000266,0.020654,2,True,10
1,NeuralNetTorch_DSTL,0.312225,0.306122,soft_log_loss,0.065258,0.011591,6.566633,0.065258,0.011591,6.566633,1,True,9
2,RandomForestMSE_DSTL,0.292934,0.306122,soft_log_loss,0.361715,0.110338,5.026858,0.361715,0.110338,5.026858,1,True,8
3,LightGBM_BAG_L1,0.269239,0.314928,accuracy,0.298056,0.033622,2.865580,0.298056,0.033622,2.865580,1,True,1
4,WeightedEnsemble_L2,0.268819,0.316973,accuracy,0.589820,0.121350,4.037856,0.002743,0.000803,0.049077,2,True,3
5,LightGBM_BAG_L1_FULL,0.267771,NaN,accuracy,0.034851,NaN,0.353911,0.034851,NaN,0.353911,1,True,4
6,WeightedEnsemble_L2_FULL,0.266513,NaN,accuracy,0.074744,NaN,0.543171,0.003639,NaN,0.049077,2,True,6
7,LightGBM_DSTL,0.219753,0.306122,soft_log_loss,0.014451,0.003829,2.357090,0.014451,0.003829,2.357090,1,True,7
8,NeuralNetTorch_BAG_L1,0.144055,0.159509,accuracy,0.289021,0.086925,1.123200,0.289021,0.086925,1.123200,1,True,2
9,NeuralNetTorch_BAG_L1_FULL,0.140491,NaN,accuracy,0.036253,NaN,0.140183,0.036253,NaN,0.140183,1,True,5


### Faster presets or hyperparameters

Instead of trying to speed up a cumbersome trained model at prediction time, if you know inference latency or memory will be an issue at the outset, then you can adjust the training process accordingly to ensure `fit()` does not produce unwieldy models.

One option is to specify more lightweight `presets`:

In [29]:
presets = ['good_quality', 'optimize_for_deployment']
predictor_light = TabularPredictor(label=label, eval_metric=metric).fit(train_data, presets=presets, time_limit=30)

No path specified. Models will be saved in: "AutogluonModels/ag-20260925_053431"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.28 GB / 12.67 GB (81.2%)
Disk Space Avail:   187.93 GB / 235.68 GB (79.7%)
Presets specified: ['good_quality', 'optimize_for_deployment']
Using hyperparameters preset: hyperparameters='light'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_ba

Another option is to specify more lightweight hyperparameters:

In [30]:
predictor_light = TabularPredictor(label=label, eval_metric=metric).fit(train_data, hyperparameters='very_light', time_limit=30)

No path specified. Models will be saved in: "AutogluonModels/ag-20260925_053503"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.37 GB / 12.67 GB (81.8%)
Disk Space Avail:   187.92 GB / 235.68 GB (79.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and 

Here you can set `hyperparameters` to either 'light', 'very_light', or 'toy' to obtain progressively smaller (but less accurate) models and predictors. Advanced users may instead try manually specifying particular models' hyperparameters in order to make them faster/smaller.

Finally, you may also exclude specific unwieldy models from being trained at all. Below we exclude models that tend to be slower (K Nearest Neighbors, Neural Network, models with custom larger-than-default  hyperparameters):

In [31]:
excluded_model_types = ['KNN', 'NN_TORCH', 'custom']
predictor_light = TabularPredictor(label=label, eval_metric=metric).fit(train_data, excluded_model_types=excluded_model_types, time_limit=30)

No path specified. Models will be saved in: "AutogluonModels/ag-20260925_053515"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.36 GB / 12.67 GB (81.8%)
Disk Space Avail:   187.92 GB / 235.68 GB (79.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and 

### (Advanced) Cache preprocessed data

If you are repeatedly predicting on the same data you can cache the preprocessed version of the data and
directly send the preprocessed data to `predictor.predict` for faster inference:

```
test_data_preprocessed = predictor.transform_features(test_data)

# The following call will be faster than a normal predict call because we are skipping the preprocessing stage.
predictions = predictor.predict(test_data_preprocessed, transform_features=False)
```


Note that this is only useful in situations where you are repeatedly predicting on the same data.
If this significantly speeds up your use-case, consider whether your current approach makes sense
or if a cache on the predictions is a better solution.

### (Advanced) Disable preprocessing

If you would rather do data preprocessing outside of TabularPredictor,
you can disable TabularPredictor's preprocessing entirely via:

```
predictor.fit(..., feature_generator=None, feature_metadata=YOUR_CUSTOM_FEATURE_METADATA)
```


Be warned that this removes ALL guardrails on data sanitization.
It is very likely that you will run into errors doing this unless you are very familiar with AutoGluon.

One instance where this can be helpful is if you have many problems
that re-use the exact same data with the exact same features. If you had 30 tasks that re-use the same features,
you could fit a `autogluon.features` feature generator once on the data, and then when you need to
predict on the 30 tasks, preprocess the data only once and then send the preprocessed data to all 30 predictors.


## If you encounter memory issues

To reduce memory usage during training, you may try each of the following strategies individually or combinations of them (these may harm accuracy):

- In `fit()`, set `excluded_model_types = ['KNN', 'XT' ,'RF']` (or some subset of these models).
- Try different `presets` in `fit()`.
- In `fit()`, set `hyperparameters = 'light'` or `hyperparameters = 'very_light'`.
- Text fields in your table require substantial memory for N-gram featurization. To mitigate this in `fit()`, you can either: (1) add `'ignore_text'` to your `presets` list (to ignore text features), or (2) specify the argument:

```
from sklearn.feature_extraction.text import CountVectorizer
from autogluon.features.generators import AutoMLPipelineFeatureGenerator
feature_generator = AutoMLPipelineFeatureGenerator(vectorizer=CountVectorizer(min_df=30, ngram_range=(1, 3), max_features=MAX_NGRAM, dtype=np.uint8))
```


for example using `MAX_NGRAM = 1000` (try various values under 10000 to reduce the number of N-gram features used to represent each text field)

In addition to reducing memory usage, many of the above strategies can also be used to reduce training times.

To reduce memory usage during inference:

- If trying to produce predictions for a large test dataset, break the test data into smaller chunks as demonstrated in [tutorials/tabular_prediction/tabular-faq.ipynb](https://github.com/gidler/autogluon-tutorials/blob/main/tutorials/tabular_prediction/tabular-faq.ipynb).

- If models have been previously persisted in memory but inference-speed is not a major concern, call `predictor.unpersist()`.

- If models have been previously persisted in memory, bagging was used in `fit()`, and inference-speed is a concern: call `predictor.refit_full()` and use one of the refit-full models for prediction (ensure this is the only model persisted in memory).



## If you encounter disk space issues

To reduce disk usage, you may try each of the following strategies individually or combinations of them:

- Make sure to delete all `predictor.path` folders from previous `fit()` runs! These can eat up your free space if you call `fit()` many times. If you didn't specify `path`, AutoGluon still automatically saved its models to a folder called: "AutogluonModels/ag-[TIMESTAMP]", where TIMESTAMP records when `fit()` was called, so make sure to also delete these folders if you run low on free space.

- Call `predictor.save_space()` to delete auxiliary files produced during `fit()`.

- Call `predictor.delete_models(models_to_keep='best', dry_run=False)` if you only intend to use this predictor for inference going forward (will delete files required for non-prediction-related functionality like `fit_summary`).

- In `fit()`, you can add `'optimize_for_deployment'` to the `presets` list, which will automatically invoke the previous two strategies after training.

- Most of the above strategies to reduce memory usage will also reduce disk usage (but may harm accuracy).


## References

The following paper describes how AutoGluon internally operates on tabular data:

Erickson et al. [AutoGluon-Tabular: Robust and Accurate AutoML for Structured Data](https://arxiv.org/abs/2003.06505). *Arxiv*, 2020.